# QFlow Toy 2D Experiment

This notebook compares QFlow against Flow Q-Learning (FQL) and Flow Behavior-Regularized Actor-Critic (FBRAC) on toy 2D datasets.

## Configuration

Modify the parameters in the next cell to customize the experiment.

In [1]:
import dotenv
%load_ext dotenv
%dotenv

In [ ]:
# Experiment parameters
FLOW_STEPS = 100
NUM_LAYERS = 4
HIDDEN_DIM = 512
BC_EPOCHS = 5000
OFFLINE_EPOCHS = 100
LEARNING_RATE = 3e-4
BATCH_SIZE = 4096
NUM_PLOTS = 10
NUM_EVALS = 512
SAVE_DIR = 'toy_experiment'

epochs = BC_EPOCHS + OFFLINE_EPOCHS

# Tasks to run: 'moons', 'two_spirals', 'swissroll', 'eight_gaussians'
TASKS = ['moons', 'two_spirals', 'swissroll']

# Guidance/regularization coefficients
ALPHAS = [5, 1, 0.5, 0.3]

# Target network settings
USE_TARGET_NETWORK = False
TAU = 0.005

# Dataset size
N_DATA = 10000

# Domain bounds
DOMAIN_MIN, DOMAIN_MAX = -4.0, 4.0

EPOCHS = BC_EPOCHS + OFFLINE_EPOCHS
PLOT_STEP = EPOCHS // NUM_PLOTS if NUM_PLOTS > 0 else None

print(f"Configuration loaded:")
print(f"  Tasks: {TASKS}")
print(f"  Alphas: {ALPHAS}")
print(f"  Epochs: {EPOCHS} (BC: {BC_EPOCHS}, Offline: {OFFLINE_EPOCHS})")
print(f"  Flow steps: {FLOW_STEPS}")
print(f"  Network: {NUM_LAYERS} layers, {HIDDEN_DIM} hidden dim")
print(f"  Target network: {USE_TARGET_NETWORK} (tau={TAU})")

In [3]:
import math
import os
import numpy as np
import jax
import jax.numpy as jnp
import jax.random as jrandom
import optax
import matplotlib.pyplot as plt
from functools import partial
from tqdm import tqdm, trange

# Project modules (under 2d_experiment/)
from networks import (
    DOMAIN_MIN, DOMAIN_MAX,
    init_mlp, apply_mlp, fourier_time_embed,
    flow_forward, flow_forward_with_time_embed, onestep_forward,
    critic_forward, inner_critic_forward,
    integrate_flow_from, integrate_flow, integrate_flow_with_time,
)
from datasets import sample_from_dataset, make_reward_fns
from agents import train_fql, train_fbrac, train_qflow, plot_training_progress

plt.rcParams['figure.figsize'] = [10, 8]
plt.rcParams['figure.dpi'] = 100

print("Imports complete!")

Imports complete!


In [4]:
task = "two_spirals"  # 'moons', 'two_spirals', 'swissroll', 'eight_gaussians'
DATASET_TYPE = task

# Oracle reward helpers for this dataset (sampler + reward functions live in datasets.py)
reward_fn_jnp, reward_fn_vec, reward_fn, plot_reward_landscape = make_reward_fns(DATASET_TYPE)

# ============================================================================
# Visualize the dataset
# ============================================================================
np.random.seed(42)
dataset_samples, dataset_rewards = sample_from_dataset(DATASET_TYPE, 2000)
dataset_rewards = dataset_rewards.flatten()  # Flatten for scatter plot c argument

plt.figure(figsize=(4, 4))
plt.scatter(dataset_samples[:, 0], dataset_samples[:, 1],
            c=dataset_rewards, cmap='plasma', s=10, alpha=0.6, edgecolors='none')
plt.title(f"Dataset Samples\nMean reward: {dataset_rewards.mean():.3f}")
plt.xlabel('Action dim 1'); plt.ylabel('Action dim 2')
plt.xlim(DOMAIN_MIN, DOMAIN_MAX)
plt.ylim(DOMAIN_MIN, DOMAIN_MAX)
plt.tight_layout()
os.makedirs(f'{SAVE_DIR}/{DATASET_TYPE}', exist_ok=True)
plt.savefig(f'{SAVE_DIR}/{DATASET_TYPE}/training_dataset.png', dpi=100, bbox_inches='tight')
plt.close()

# ============================================================================
# Generate Training Data
# ============================================================================
# Dataset: clean samples x_1 from the dataset distribution
# Reward: R(x_1) explicitly returned by the reference sampler
np.random.seed(0)
x_1_data, rewards_data = sample_from_dataset(DATASET_TYPE, N_DATA)
rewards_data = np.array(rewards_data).flatten()

print(f"Generated {len(x_1_data)} samples from '{DATASET_TYPE}' "
      f"(mean reward: {rewards_data.mean():.3f})")

Generated 10000 samples from 'two_spirals' (mean reward: 0.374)


In [5]:
# ============================================================================
# Training functions (train_fql / train_fbrac / train_qflow) and the
# plot_training_progress helper now live in agents.py and are imported above.
# Network definitions live in networks.py; dataset/reward code in datasets.py.
# ============================================================================
print("train_fql / train_fbrac / train_qflow imported from agents.py")

train_fql / train_fbrac / train_qflow imported from agents.py


## Training Runs

In [6]:
fql_results = []
ALPHAS=[1, 0.5]

for ALPHA in ALPHAS:
    fql_flow_params, fql_policy_params, fql_critic_params, fql_bc_policy_params, fql_history = train_fql(
        x_1_data, rewards_data, 
        epochs=EPOCHS,
        lr=LEARNING_RATE,
        alpha=ALPHA,  # Weight for Q-maximization
        hidden=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        flow_steps=FLOW_STEPS,
        plot_every=PLOT_STEP,
        use_target_network=USE_TARGET_NETWORK,
        tau=TAU,
        batch_size=BATCH_SIZE,
        bc_epochs=BC_EPOCHS,
        num_evals=NUM_EVALS,
        save_dir=SAVE_DIR,
        dataset_type=DATASET_TYPE,
    )
    fql_results.append((ALPHA, fql_policy_params, fql_bc_policy_params))


# Sort alphas in decreasing order for plotting
fql_results_sorted = sorted(fql_results, key=lambda x: x[0], reverse=True)
n_cols = 1 + len(fql_results_sorted)
fig, axes = plt.subplots(1, n_cols, figsize=(3 * n_cols, 3))
if n_cols == 1:
    axes = [axes]

# BC baseline: Output of BC one-step policy
ax0 = axes[0]
z_eval = jrandom.normal(jrandom.PRNGKey(999), (NUM_EVALS, 2))
# Use the first bc_policy_params available (from largest alpha, due to sorting reverse)
bc_policy_params = fql_results_sorted[0][2]
bc_actions = np.array(onestep_forward(bc_policy_params, z_eval))
ax0.scatter(
    bc_actions[:, 0],
    bc_actions[:, 1],
    s=10,
    alpha=0.6,
    edgecolors='none'
)
ax0.set_title("BC", fontsize=15, fontstyle='italic')
ax0.set_xlim(DOMAIN_MIN, DOMAIN_MAX); ax0.set_ylim(DOMAIN_MIN, DOMAIN_MAX)
ax0.set_aspect('equal')

# Offline RL one-step policies per alpha (no coloring)
for i, (alpha, policy_params, _) in enumerate(fql_results_sorted):
    actions = np.array(onestep_forward(policy_params, z_eval))
    ax = axes[i + 1]
    ax.scatter(
        actions[:, 0],
        actions[:, 1],
        s=10,
        alpha=0.6,
        edgecolors='none'
    )
    ax.set_title(f'α={alpha}', fontsize=15, fontstyle='italic')
    ax.set_xlim(DOMAIN_MIN, DOMAIN_MAX); ax.set_ylim(DOMAIN_MIN, DOMAIN_MAX)
    ax.set_aspect('equal')

plt.show()

2026-06-22 22:16:31.611726: E external/xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


RuntimeError: Unable to initialize backend 'cuda': FAILED_PRECONDITION: No visible GPU devices. (you may need to uninstall the failing plugin package, or set JAX_PLATFORMS=cpu to skip this backend.)

In [ ]:
fbrac_results = []

for ALPHA in ALPHAS:
    fbrac_outer_flow_params, fbrac_outer_critic_params, fbrac_outer_bc_policy_params, fbrac_outer_history = train_fbrac(
        x_1_data, rewards_data, 
        epochs=EPOCHS, 
        lr=LEARNING_RATE, 
        alpha=ALPHA,  # BC regularization weight
        hidden=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        flow_steps=FLOW_STEPS,
        plot_every=PLOT_STEP,
        use_target_network=USE_TARGET_NETWORK,
        tau=TAU,
        batch_size=BATCH_SIZE,
        bc_epochs=BC_EPOCHS,
        num_evals=NUM_EVALS,
        save_dir=SAVE_DIR,
        dataset_type=DATASET_TYPE,
    )
    fbrac_results.append((ALPHA, fbrac_outer_flow_params, fbrac_outer_bc_policy_params))


# Sort alphas in decreasing order for plotting
fbrac_results_sorted = sorted(fbrac_results, key=lambda x: x[0], reverse=True)
n_cols = 1 + len(fbrac_results_sorted)
fig, axes = plt.subplots(1, n_cols, figsize=(3 * n_cols, 3))
if n_cols == 1:
    axes = [axes]

# BC baseline: output of BC-trained flow policy (as stored, shape matches expected for flow)
ax0 = axes[0]
z_eval = jrandom.normal(jrandom.PRNGKey(999), (NUM_EVALS, 2))
bc_flow_params = fbrac_results_sorted[0][2]
# Defensive: if bc_flow_params is a tuple (from possible storage / wrapping), extract first
if isinstance(bc_flow_params, tuple):
    bc_flow_params_to_use = bc_flow_params[0]
else:
    bc_flow_params_to_use = bc_flow_params
try:
    bc_actions = np.array(integrate_flow(bc_flow_params_to_use, z_eval, steps=FLOW_STEPS))
except TypeError:
    # Fall back on utility wrapper (handles possible legacy flow param signatures)
    bc_actions = np.array(integrate_flow_with_time(bc_flow_params_to_use, z_eval, steps=FLOW_STEPS))
ax0.scatter(
    bc_actions[:, 0],
    bc_actions[:, 1],
    s=10,
    alpha=0.6,
    edgecolors='none'
)
ax0.set_title("BC", fontsize=15, fontstyle='italic')
ax0.set_xlim(DOMAIN_MIN, DOMAIN_MAX); ax0.set_ylim(DOMAIN_MIN, DOMAIN_MAX)
ax0.set_aspect('equal')

# Offline RL flow policies per alpha (no coloring)
for i, (alpha, flow_params, _) in enumerate(fbrac_results_sorted):
    # Handle tuple wrapping as above
    if isinstance(flow_params, tuple):
        flow_params_to_use = flow_params[0]
    else:
        flow_params_to_use = flow_params
    try:
        actions = np.array(integrate_flow(flow_params_to_use, z_eval, steps=FLOW_STEPS))
    except TypeError:
        actions = np.array(integrate_flow_with_time(flow_params_to_use, z_eval, steps=FLOW_STEPS))
    ax = axes[i + 1]
    ax.scatter(
        actions[:, 0],
        actions[:, 1],
        s=10,
        alpha=0.6,
        edgecolors='none'
    )
    ax.set_title(f'α={alpha}', fontsize=15, fontstyle='italic')
    ax.set_xlim(DOMAIN_MIN, DOMAIN_MAX); ax.set_ylim(DOMAIN_MIN, DOMAIN_MAX)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
# Experiment parameters
FLOW_STEPS = 100
NUM_LAYERS = 4
HIDDEN_DIM = 512
BC_EPOCHS = 5000
OFFLINE_EPOCHS = 100
LEARNING_RATE = 3e-4
BATCH_SIZE = 4096
NUM_PLOTS = 2
NUM_EVALS = 512

epochs = BC_EPOCHS + OFFLINE_EPOCHS
ALPHAS = [0.5, 0.3]
USE_TARGET_NETWORK = False
TAU = 0.005

EPOCHS = BC_EPOCHS + OFFLINE_EPOCHS
PLOT_STEP = EPOCHS // NUM_PLOTS if NUM_PLOTS > 0 else None


qflow_results = []

for ALPHA in ALPHAS:
    qflow_flow_params, qflow_critic_params, qflow_bc_policy_params, qflow_history = train_qflow(
        x_1_data, rewards_data, 
        epochs=EPOCHS,
        lr=LEARNING_RATE,
        alpha=ALPHA,
        flow_steps=FLOW_STEPS,
        plot_every=PLOT_STEP,
        use_time_embed=False,
        time_embed_dim=16,
        hidden=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        use_target_network=USE_TARGET_NETWORK,
        tau=TAU,
        batch_size=BATCH_SIZE,
        bc_epochs=BC_EPOCHS,
        num_evals=NUM_EVALS,
        save_dir=SAVE_DIR,
        dataset_type=DATASET_TYPE,
    )
    import pickle, os
    weights_dir = f'{SAVE_DIR}/{DATASET_TYPE}/qflow'
    os.makedirs(weights_dir, exist_ok=True)
    weights_path = (f'{weights_dir}/weights_alpha{ALPHA}_flow{FLOW_STEPS}_layers{NUM_LAYERS}'
                    f'_hidden{HIDDEN_DIM}_data{N_DATA}_epochs{EPOCHS}.pkl')
    with open(weights_path, 'wb') as f_w:
        pickle.dump({'flow_params': qflow_flow_params,
                     'inner_critic_params': qflow_critic_params,
                     'bc_policy_params': qflow_bc_policy_params}, f_w)
    print(f'Weights saved → {weights_path}')

    qflow_results.append((ALPHA, qflow_flow_params, qflow_bc_policy_params))

# Sort alphas in decreasing order for plotting
qflow_results_sorted = sorted(qflow_results, key=lambda x: x[0], reverse=True)
n_cols = 1 + len(qflow_results_sorted)
fig, axes = plt.subplots(1, n_cols, figsize=(3 * n_cols, 3))
if n_cols == 1:
    axes = [axes]

# BC baseline: output of BC-trained flow policy
ax0 = axes[0]
z_eval = jrandom.normal(jrandom.PRNGKey(999), (NUM_EVALS, 2))
bc_flow_params = qflow_results_sorted[0][2]
if isinstance(bc_flow_params, tuple):
    bc_flow_params_to_use = bc_flow_params[0]
else:
    bc_flow_params_to_use = bc_flow_params
try:
    bc_actions = np.array(integrate_flow(bc_flow_params_to_use, z_eval, steps=FLOW_STEPS))
except TypeError:
    bc_actions = np.array(integrate_flow_with_time(bc_flow_params_to_use, z_eval, steps=FLOW_STEPS))
ax0.scatter(
    bc_actions[:, 0],
    bc_actions[:, 1],
    s=10,
    alpha=0.6,
    edgecolors='none'
)
ax0.set_title("BC", fontsize=15, fontstyle='italic')
ax0.set_xlim(DOMAIN_MIN, DOMAIN_MAX); ax0.set_ylim(DOMAIN_MIN, DOMAIN_MAX)
ax0.set_aspect('equal')

# Offline RL flow policies per alpha
for i, (alpha, flow_params, _) in enumerate(qflow_results_sorted):
    if isinstance(flow_params, tuple):
        flow_params_to_use = flow_params[0]
    else:
        flow_params_to_use = flow_params
    try:
        actions = np.array(integrate_flow(flow_params_to_use, z_eval, steps=FLOW_STEPS))
    except TypeError:
        actions = np.array(integrate_flow_with_time(flow_params_to_use, z_eval, steps=FLOW_STEPS))
    ax = axes[i + 1]
    ax.scatter(
        actions[:, 0],
        actions[:, 1],
        s=10,
        alpha=0.6,
        edgecolors='none'
    )
    ax.set_title(f'α={alpha}', fontsize=15, fontstyle='italic')
    ax.set_xlim(DOMAIN_MIN, DOMAIN_MAX); ax.set_ylim(DOMAIN_MIN, DOMAIN_MAX)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()